# Chapter 9: Text Processing and Multiclass Classification

## Ringkasan Bab & Penjelasan Teoretis
Algoritma machine learning membutuhkan representasi numerik terstruktur, sehingga data teks bebas (*unstructured text*) harus melalui proses tokenisasi, pembersihan *stop words*, dan vektorisasi sebelum dapat diklasifikasikan.

### 1. Teknik Vektorisasi Teks & Ekstraksi Fitur N-Grams
* **Bag-of-Words (`CountVectorizer`)**: Mengonversi koleksi dokumen teks menjadi matriks frekuensi kemunculan kata (token). Mengabaikan urutan kata namun efektif menangkap kosakata kunci.
* **TF-IDF (`TfidfVectorizer`)**: *Term Frequency-Inverse Document Frequency* memberi bobot tinggi pada kata yang sering muncul di suatu dokumen spesifik namun jarang muncul di seluruh korpus, sehingga menekan pengaruh kata-kata umum:
  $$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \left(\log \frac{1 + n}{1 + \text{DF}(t)} + 1\right)$$
* **Ekstraksi Fitur N-Grams (`ngram_range`)**: Menangkap konteks frasa berurutan dengan menggabungkan $N$ kata yang berdekatan (misalnya *unigram* $(1, 1)$ dan *bigram* $(1, 2)$ seperti `"machine learning"`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Korpus Teks Sintetis Multikelas (3 Topik: Technology, Sports, Finance)
corpus = [
    "Python machine learning algorithms and artificial intelligence neural networks",
    "Deep learning models GPU computing software engineering and data science",
    "Cloud computing cybersecurity servers database programming and API development",
    "Software developers build machine learning pipelines using scikit-learn in Python",
    "Artificial intelligence and automated data processing systems in modern tech",
    "Football championship tournament match goal striker and team victory",
    "Basketball players playoffs court score championship and coach strategy",
    "Olympic athletes marathon running track medal and world record training",
    "Tennis grand slam match tournament winner serve and athletic performance",
    "Soccer league striker scored hat-trick goal in the final stadium match",
    "Stock market investment portfolio dividends trading and financial returns",
    "Central bank interest rates inflation monetary policy and economic growth",
    "Corporate earnings revenue profit margin quarterly report and shareholders",
    "Cryptocurrency blockchain assets hedge fund capital market and bond yields",
    "Banking credit loans mortgage interest rates and global financial markets"
] * 10  # Duplikasi dengan variasi untuk simulasi pembagian train/test

labels = ([0]*5 + [1]*5 + [2]*5) * 10
target_names = ['Technology', 'Sports', 'Finance']

# 1. CountVectorizer (Bag-of-Words)
count_vec = CountVectorizer(stop_words='english')
X_counts = count_vec.fit_transform(corpus[:6])
print("Contoh Kosakata CountVectorizer:", list(count_vec.vocabulary_.keys())[:10])
print("Dimensi Matriks CountVectorizer:", X_counts.shape)

# 2. TfidfVectorizer dengan Ekstraksi N-Grams (Unigram + Bigram)
tfidf_vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=50)
X_tfidf = tfidf_vec.fit_transform(corpus)
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out())
print("\nSampel Matriks TF-IDF dengan Bigram:")
display(df_tfidf.head(5))

### 2. Implementasi & Evaluasi Model Klasifikasi Teks Multikelas
Model seperti **Multinomial Naive Bayes (`MultinomialNB`)** dan **Logistic Regression** (dengan strategi *One-vs-Rest* / multikelas) sangat cocok dipadukan dengan matriks jarang (*sparse matrix*) hasil `TfidfVectorizer` di dalam sebuah `Pipeline`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(corpus, labels, test_size=0.3, random_state=2024, stratify=labels)

# Pipeline Klasifikasi Teks Multikelas: TF-IDF + Multinomial Naive Bayes
text_clf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('clf', OneVsRestClassifier(MultinomialNB(alpha=0.5)))
])

text_clf_pipeline.fit(X_train, y_train)
y_pred = text_clf_pipeline.predict(X_test)

print(f"Multiclass Text Classifier Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
report_df = pd.DataFrame(classification_report(y_test, y_pred, target_names=target_names, output_dict=True)).transpose()
display(report_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted Topic')
plt.ylabel('True Topic')
plt.title('Text Classification Confusion Matrix')
plt.show()